# Post 006 — Unsupervised Discovery: K-Means, DBSCAN & PCA
## Dataset B: Wafer Defect Pattern Clustering (Post-Silicon Validation)

**AI Engineering Lab Series | Era 1: Classic Machine Learning**

---

In semiconductor manufacturing, wafers come out of fabrication with various defect patterns. Some patterns indicate systematic process issues (e.g., edge ring defects from uneven deposition), while others are random. Identifying these patterns automatically — without manual labeling — is exactly what unsupervised learning was built for.

This notebook applies K-Means and DBSCAN to cluster wafer defect signatures from electrical measurements, helping engineers quickly identify which process step is causing yield loss.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded')

## 1. Load and Explore the Wafer Dataset

Each row represents one die on a wafer. Features include electrical test measurements: leakage current, threshold voltage, ring oscillator frequency (process speed proxy), contact resistance, and spatial position on the wafer (die_x, die_y). The `defect_type` column is our ground truth for evaluation only.

In [ ]:
df = pd.read_csv('../data/wafer_defect_patterns.csv')
print(f'Shape: {df.shape}')
print(f'\nDefect type distribution:')
print(df['defect_type'].value_counts())
df.head()

In [ ]:
# Visualize the wafer map — spatial distribution of defect types
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

defect_types = df['defect_type'].unique()
colors = plt.cm.Set1(np.linspace(0, 1, len(defect_types)))

for dt, col in zip(defect_types, colors):
    mask = df['defect_type'] == dt
    axes[0].scatter(df.loc[mask, 'die_x'], df.loc[mask, 'die_y'], 
                   c=[col], label=dt, alpha=0.6, s=20)
axes[0].set_title('Wafer Map — True Defect Types')
axes[0].set_xlabel('Die X Position')
axes[0].set_ylabel('Die Y Position')
axes[0].legend(fontsize=7)
axes[0].set_aspect('equal')

# Electrical feature correlation
feature_cols = [c for c in df.columns if c not in ['defect_type']]
corr = df[feature_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.show()

## 2. Preprocessing and PCA

Wafer electrical measurements span very different scales — leakage current is in nanoamps while ring oscillator frequency is in MHz. Scaling is critical. We then apply PCA to understand which electrical signatures most differentiate the defect types.

In [ ]:
X = df[feature_cols].values
y_true = df['defect_type'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'PC1 explains: {pca.explained_variance_ratio_[0]:.1%}')
print(f'PC2 explains: {pca.explained_variance_ratio_[1]:.1%}')
print(f'Total:        {pca.explained_variance_ratio_.sum():.1%}')

# PCA loadings — what drives each component?
loadings_df = pd.DataFrame(pca.components_.T, index=feature_cols, columns=['PC1', 'PC2'])
print('\nTop features driving PC1 (process speed / electrical signature):')
print(loadings_df['PC1'].abs().sort_values(ascending=False).head(4))

## 3. K-Means: Finding Defect Clusters

We know from domain knowledge there are roughly 5-7 defect types. But let's pretend we don't and use the Elbow Method to discover this from the data itself. This is the real-world scenario: a new fab process with unknown failure modes.

In [ ]:
# Elbow + Silhouette analysis
inertias, sil_scores = [], []
K_range = range(2, 12)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_, sample_size=2000))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(K_range, inertias, 'bo-', linewidth=2)
ax1.set_xlabel('K'); ax1.set_ylabel('Inertia'); ax1.set_title('Elbow Method — Wafer Defects')
ax2.plot(K_range, sil_scores, 'go-', linewidth=2)
best_k = K_range[sil_scores.index(max(sil_scores))]
ax2.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
ax2.set_xlabel('K'); ax2.set_ylabel('Silhouette Score'); ax2.set_title('Silhouette Score')
ax2.legend()
plt.tight_layout()
plt.show()
print(f'Optimal K from silhouette: {best_k}')

In [ ]:
# Fit K-Means with optimal K
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
km_labels = kmeans.fit_predict(X_scaled)

ari = adjusted_rand_score(y_true, km_labels)
sil = silhouette_score(X_scaled, km_labels)
print(f'K-Means (K={best_k}): ARI={ari:.3f}, Silhouette={sil:.3f}')

# Wafer map with K-Means clusters
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

scatter = ax1.scatter(df['die_x'], df['die_y'], c=km_labels, cmap='tab10', alpha=0.6, s=20)
ax1.set_title(f'K-Means Clusters on Wafer Map (K={best_k}, ARI={ari:.3f})')
ax1.set_xlabel('Die X'); ax1.set_ylabel('Die Y')
ax1.set_aspect('equal')
plt.colorbar(scatter, ax=ax1, label='Cluster')

for dt, col in zip(defect_types, colors):
    mask = y_true == dt
    ax2.scatter(df.loc[mask, 'die_x'], df.loc[mask, 'die_y'], c=[col], label=dt, alpha=0.6, s=20)
ax2.set_title('True Defect Types')
ax2.set_xlabel('Die X'); ax2.set_ylabel('Die Y')
ax2.legend(fontsize=7)
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

## 4. DBSCAN: Detecting Spatial Defect Clusters

DBSCAN is particularly powerful for wafer maps because defects often form spatially contiguous regions (e.g., an edge ring, a scratch line, a center spot). DBSCAN naturally finds these irregular shapes, while K-Means would force them into spherical clusters.

In [ ]:
# DBSCAN on spatial coordinates only (die_x, die_y)
spatial_cols = ['die_x', 'die_y']
X_spatial = scaler.fit_transform(df[spatial_cols].values)

dbscan_spatial = DBSCAN(eps=0.3, min_samples=10)
db_spatial_labels = dbscan_spatial.fit_predict(X_spatial)

n_clusters = len(set(db_spatial_labels)) - (1 if -1 in db_spatial_labels else 0)
n_noise = (db_spatial_labels == -1).sum()

print(f'DBSCAN (spatial): {n_clusters} clusters, {n_noise} noise points ({n_noise/len(db_spatial_labels):.1%})')

fig, ax = plt.subplots(figsize=(8, 8))
unique_labels = sorted(set(db_spatial_labels))
cmap = plt.cm.get_cmap('tab20', max(len(unique_labels), 1))
for i, label in enumerate(unique_labels):
    mask = db_spatial_labels == label
    if label == -1:
        ax.scatter(df.loc[mask, 'die_x'], df.loc[mask, 'die_y'], c='lightgray', alpha=0.3, s=15, label='Noise')
    else:
        ax.scatter(df.loc[mask, 'die_x'], df.loc[mask, 'die_y'], c=[cmap(i)], alpha=0.7, s=20, label=f'Region {label}')
ax.set_title(f'DBSCAN Spatial Clustering — {n_clusters} defect regions detected')
ax.set_xlabel('Die X'); ax.set_ylabel('Die Y')
ax.set_aspect('equal')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 5. Cluster Profile Analysis: What Does Each Cluster Mean?

After clustering, the most important step is **interpretation**. What electrical signature defines each cluster? This maps clusters back to physical root causes — the ultimate goal for a validation engineer.

In [ ]:
# Cluster profiles from K-Means
df_analysis = df.copy()
df_analysis['kmeans_cluster'] = km_labels

cluster_profiles = df_analysis.groupby('kmeans_cluster')[feature_cols].mean()

# Normalize for heatmap
normalized = (cluster_profiles - cluster_profiles.mean()) / cluster_profiles.std()

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(normalized, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax,
            cbar_kws={'label': 'Z-score (deviation from mean)'})
ax.set_title('K-Means Cluster Profiles — Electrical Signatures per Cluster')
ax.set_xlabel('Electrical Feature')
ax.set_ylabel('Cluster ID')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('\nInterpretation guide:')
print('  High leakage + low Vth → Gate oxide defect (thin oxide)')
print('  High contact resistance → Metal interconnect defect')
print('  Low ring oscillator freq → Slow process corner or contamination')
print('  Edge-concentrated → Deposition uniformity issue')

## 6. Summary

This notebook demonstrated how unsupervised learning can automatically discover defect patterns in wafer data that would take a validation engineer hours to identify manually.

| Method | Best For | Wafer Application |
|---|---|---|
| **K-Means** | Known number of defect types, electrical signatures | Classifying dies by electrical failure mode |
| **DBSCAN** | Unknown number of clusters, spatial patterns | Finding contiguous defect regions on wafer map |
| **PCA** | High-dimensional electrical data, visualization | Reducing 20+ test parameters to 2D for inspection |

**The key insight**: Unsupervised learning does not replace the engineer — it amplifies them. Instead of manually reviewing 15,000 die results, the engineer reviews 5-7 cluster profiles and maps them to root causes.